# **🛡️ Assignment 1: Please solve all the problems in your copy version of assignment for each one of you, and Submit your solution in the Notebook format.**
###**Your submission will be subjected to an AI detector. Do not use AI tools to generate your work, or you may lose the full grade for the assignment.**




---

## **Instructions:**
- Answer each question by writing the requested Python code.
- Make sure you run your code on the data and there is output.
- Each question focuses on a different data Cleaning concept.
- Marks will be awarded for correctness, clarity, and following best practices.

---

## **1. Data Profiling**
**Task:**
#####Write Python code to **Profile** the Data uplouded with this notebook using any of the following libraries; **ydata_profiling** or **sweetviz** and **Submit the HTML file with the notebook.**
---
## **2. Data Validation**
**Task:**  
Write a Python code that execute the following:
  1. Using **pandera** Library to extract the inferred_schema of this data.
  2. Using **great_expectations** library to:
      - Validate the null values for all columns.

      - Validate the Values of Rating Column to be between 1 to 10.

      - Validate the values of Movies Column to be unique.

      - X = ['Animation', Romance', Action', Adventure', Drama', Horror', 'Thriller', 'Crime', 'Mystery', 'Comedy'].
      
        Validate that GENRE Column values be in set values of X.

      - Save the expectation suite and add the above checkpoints, then print its result.

  3. Using **cerberus** Library to make a Validator to extract the rows that achive that it has Rating above 9 and it's Greans contaian one of these ['Animation', Action', Adventure', Drama', 'Mystery', 'Comedy'] and print them from the data.
---

## **3. Data Cleaning (Missing Values, Duplicates and Outliers)**
**Task:**  
Write Python code to clean the same data that you review its problems through the above profiling task, Following the next pipline in order:
1. **Convert the wrong Format** for these columns (Year, Votes).
2. **Drop the Gross** column from the dataset due to much of **missing**.
3. **Drop the missing** values **subset from GENRE** column.
4. Choose one of the known methods **(Mean-Median-Mode)** to **fill the missing values in the other columns**.
5. **Drop the duplicates**.
6. **Do suitable Scaling**.
7. **Check for Outliers and Solve it**.
---






In [ ]:
# install libraries
!pip install pandas numpy ydata-profiling pandera great-expectations cerberus scikit-learn

In [ ]:
import pandas as pd

# load dataset
df = pd.read_csv('/content/movies.csv')

# preview data
print("First 5 rows:")
print(df.head())

print("\nShape of dataset:")
print(df.shape)

print("\nColumns:")
print(df.columns)

In [ ]:
# check data types
print("\nData types:")
print(df.dtypes)

In [ ]:
# check missing values
print("\nMissing values:")
print(df.isnull().sum())

# 1. Data Profiling

In [ ]:
from ydata_profiling import ProfileReport

# create profile report
profile = ProfileReport(df, title="Movies Dataset Report")

# save as HTML
profile.to_file("movies_report.html")

print("Profile report generated successfully!")

# 2. Data Validation

In [ ]:
# clean column names
df.columns = df.columns.str.strip()

print("Columns:")
print(df.columns)

In [ ]:
import pandera as pa

schema = pa.infer_schema(df)

print("\nPandera Inferred Schema:")
print(schema)

In [ ]:
import great_expectations as gx
from great_expectations.validator.validator import Validator
from great_expectations.core.batch import Batch
from great_expectations.execution_engine import PandasExecutionEngine

batch = Batch(data=df)
engine = PandasExecutionEngine()

validator = Validator(
    execution_engine=engine,
    batches=[batch]
)

In [ ]:
# 1. Check null values
for col in df.columns:
    validator.expect_column_values_to_not_be_null(col)

# 2. Rating between 1 and 10
validator.expect_column_values_to_be_between(
    "RATING", min_value=1, max_value=10
)

# 3. MOVIES must be unique
validator.expect_column_values_to_be_unique("MOVIES")

# 4. GENRE must contain allowed values
X = ['Animation', 'Romance', 'Action', 'Adventure', 'Drama',
     'Horror', 'Thriller', 'Crime', 'Mystery', 'Comedy']

validator.expect_column_values_to_match_regex(
    "GENRE",
    "|".join(X)
)

In [ ]:
suite = validator.get_expectation_suite()

print("\nExpectation Suite:")
print(suite)

In [ ]:
results = validator.validate()

print("\nValidation Results:")
print(results)

In [ ]:
from cerberus import Validator
import pandas as pd

# make sure rating is numeric
df['RATING'] = pd.to_numeric(df['RATING'], errors='coerce')

allowed_genres = ['Animation', 'Action', 'Adventure',
                  'Drama', 'Mystery', 'Comedy']

filtered_rows = []

for row in df.to_dict(orient='records'):

    # condition: rating > 9
    if row['RATING'] is not None and not pd.isna(row['RATING']) and row['RATING'] > 9:

        genre = str(row['GENRE'])

        for g in allowed_genres:
            if g in genre:
                filtered_rows.append(row)
                break

filtered_df = pd.DataFrame(filtered_rows)

print("\nFiltered Rows (Rating > 9):")
print(filtered_df)

In [ ]:
print("Max rating:", df['RATING'].max())

# 3. Data Cleaning

In [ ]:
# clean YEAR
df['YEAR'] = pd.to_numeric(df['YEAR'], errors='coerce')

# clean VOTES (remove commas first)
df['VOTES'] = df['VOTES'].astype(str).str.replace(',', '')
df['VOTES'] = pd.to_numeric(df['VOTES'], errors='coerce')

print("After fixing formats:")
print(df[['YEAR', 'VOTES']].head())

In [ ]:
df.drop(columns=['Gross'], inplace=True)

print("Columns after dropping Gross:")
print(df.columns)

In [ ]:
df.dropna(subset=['GENRE'], inplace=True)

print("After dropping missing GENRE:")
print(df.shape)

In [ ]:
# numeric columns → median
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# categorical columns → mode
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("Missing values after filling:")
print(df.isnull().sum())

In [ ]:
df.drop_duplicates(inplace=True)

print("After removing duplicates:")
print(df.shape)

In [ ]:
# select numeric columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)
IQR = Q3 - Q1

# remove outliers
df = df[~((df[num_cols] < (Q1 - 1.5 * IQR)) |
          (df[num_cols] > (Q3 + 1.5 * IQR))).any(axis=1)]

print("After removing outliers:")
print(df.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = df.select_dtypes(include=['int64', 'float64']).columns

df[num_cols] = scaler.fit_transform(df[num_cols])

print("After scaling:")
print(df.head())

In [ ]:
print("\nFinal Data Info:")
df.info()

print("\nFinal Preview:")
df.head()